# xenafan.com: parsed sample for review

Notebook version of `fanfic-sample-xenafan`. Parses the saved copy of `titles.html` (about 1,450 stories) and lets you inspect the records.

**Network:** none if `verify_xenafan.ipynb` (or `fanfic-verify-xenafan`) has already saved `titles.html`. Otherwise it is downloaded first (1 request, plus robots.txt).

Known limitation: 8 stories are split across two index entries whose second half is marked "(cont'd)". They are kept as two records rather than merged; the "Same-site collisions" section lists them.

In [ ]:
import os
import pathlib
import sys

# Find the repository root (the folder holding pyproject.toml) whether this
# notebook was opened from notebooks/ or from the root, and work from there,
# so saved pages and outputs always go to <repo>/data/ (or FANFIC_DATA_DIR).
REPO_ROOT = pathlib.Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Open this notebook from inside the repository folder.")
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))  # works without `pip install -e .`
REPO_ROOT

## Settings

`CONTACT_EMAIL` is sent with every request (in the User-Agent) so site operators can reach whoever is crawling. Use your own address. Leave it blank to be asked the first time a download is needed, or set the `FANFIC_CONTACT_EMAIL` environment variable instead.

Only used if `titles.html` has not been saved yet.

In [ ]:
CONTACT_EMAIL = ""  # your email address

In [ ]:
import dataclasses
import json
from collections import defaultdict

from fanfic_scraper.discover.xenafan import discover_xenafan
from fanfic_scraper.fetch import XENAFAN, build_default_fetcher
from fanfic_scraper.parse.xenafan import parse_xenafan_titles

fetcher = build_default_fetcher(XENAFAN, contact_email=CONTACT_EMAIL or None)
cached = discover_xenafan(fetcher)
records = parse_xenafan_titles(cached.body)

print(f"Parsed {len(records)} records from xenafan titles.html")

## Sample records

First 10 parsed records, full JSON.

In [ ]:
print(json.dumps([dataclasses.asdict(r) for r in records[:10]], indent=2, ensure_ascii=False))

## Search by title or author

Edit the query and re-run.

In [ ]:
def search(query, by="title"):
    q = query.lower()
    if by == "title":
        return [r for r in records if q in r.title.lower()]
    if by == "author":
        return [r for r in records if any(q in a.lower() for a in r.authors)]
    raise ValueError("by must be 'title' or 'author'")


for r in search("Ab Initio"):
    print(f"{r.title!r} by {r.authors} - {len(r.parts)} parts, notes={r.notes!r}")

## Same-site collisions

Groups records by `(title_normalized, authors)`. More than one record in a group is either a story split across two entries (the "(cont'd)" case) or a duplicate listing on the site itself. Neither is merged at parse time.

In [ ]:
groups = defaultdict(list)
for r in records:
    groups[(r.title_normalized, tuple(r.authors))].append(r)

collisions = {key: recs for key, recs in groups.items() if len(recs) > 1}
print(f"{len(collisions)} title+author collisions among {len(records)} parsed records\n")
for (title, authors), recs in collisions.items():
    print(f"{title!r} by {list(authors)}:")
    for r in recs:
        print(f"    source_id={r.source_id!r}  notes={r.notes!r}  parts={len(r.parts)}")

## Removed / lost stories and external links

`status="removed"` marks entries such as "(file lost to time)"; these still have an author. `external_url` is set for stories that link off-site via `storylinks.html`.

In [ ]:
removed = [r for r in records if r.status == "removed"]
external = [r for r in records if r.external_url]

print(f"{len(removed)} removed/lost record(s):")
for r in removed:
    print(f"    {r.title!r} by {r.authors} - notes={r.notes!r}")

print(f"\n{len(external)} external record(s):")
for r in external:
    print(f"    {r.title!r} by {r.authors} - external_url={r.external_url!r}")

## Write full output to disk

The same file `fanfic-sample-xenafan` writes: all parsed records.

In [ ]:
from fanfic_scraper.fetch import default_data_dir

out_path = default_data_dir() / "xenafan_sample_output.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump([dataclasses.asdict(r) for r in records], f, indent=2, ensure_ascii=False)
print(f"Wrote {len(records)} records to {out_path}")